# 🚀 LifeMate ZAIC: Pipeline Huấn Luyện ASR Tiếng Việt (<1.5B Params) & BGE-M3 Embedding

**Mục tiêu đạt được:**
- Mô hình ASR: `openai/whisper-large-v3-turbo` (809M tham số < 1.5B, tốc độ suy luận nhanh gấp 8 lần).
- Độ chính xác mục tiêu: **WER < 3%** trên các câu lệnh tiếng Việt và chêm tiếng Anh (Code-Switching).
- Thời gian chạy ứng dụng: **8 - 10 tiếng liên tục** trên Google Colab (T4 / A100 GPU).
- Cơ chế: Tự động mount Google Drive để lưu checkpoint mỗi 250 bước (chống mất dữ liệu khi rớt kết nối).

## 1. Kết Nối Google Drive & Cấu Hình Môi Trường

In [ ]:
# Mount Google Drive để lưu trữ Checkpoint suốt 10 tiếng chạy
from google.colab import drive
import os

drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/LifeMate_ZAIC_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Đã sẵn sàng thư mục lưu checkpoint tại: {CHECKPOINT_DIR}")

## 2. Snippet Chống Ngắt Kết Nối (Keep-Alive Cho Colab Khi Bạn Đi Vắng)
*Nhấn `F12` trong trình duyệt, chuyển sang tab **Console**, dán đoạn code dưới đây và nhấn Enter để trình duyệt không bị idle:*
```javascript
function ConnectButton(){
    console.log("Keep alive click");
    document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click();
}
setInterval(ConnectButton, 60000);
```

## 3. Cài Đặt Thư Viện Tối Ưu Hóa VRAM

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers datasets evaluate jiwer accelerate peft bitsandbytes librosa soundfile sentence-transformers

## 4. Bộ Chuẩn Hóa Văn Bản Tiếng Việt (Bắt Buộc Để WER < 3%)

In [ ]:
import re
import unicodedata

def prepare_for_wer(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    # 1. Unicode chuẩn dựng sẵn NFC
    text = unicodedata.normalize("NFC", text).lower()
    # 2. Xóa các nhãn tag nhiễu
    text = re.sub(r"<[^>]+>|\[[^\]]+\]|\([^\)]+\)", " ", text)
    # 3. Loại bỏ từ đệm
    text = re.sub(r"\b(à|ừm|ờ|ơ|uh|um|ehm)\b", " ", text)
    # 4. Loại bỏ dấu câu, chỉ giữ ký tự chữ và số
    text = re.sub(r"[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print("Test chuẩn hóa:", prepare_for_wer("Mai schedule meeting lúc 8 giờ nha... <noise>"))

## 5. Tải Dataset & Chuẩn Bị Dữ Liệu
- Mặc định sử dụng `google/fleurs` (vi_vn) kết hợp các nguồn mở trên Hugging Face.
- Nếu có tập dữ liệu Zalo LifeMate, bạn đặt vào Google Drive và điền đường dẫn CSV bên dưới.

In [ ]:
from datasets import load_dataset, Audio

print("📥 Tải Google FLEURS vi_vn chuẩn quốc tế...")
fleurs = load_dataset("google/fleurs", "vi_vn", trust_remote_code=True)
fleurs = fleurs.cast_column("audio", Audio(sampling_rate=16000))
fleurs = fleurs.rename_column("transcription", "sentence")

print(f"Tập Train: {len(fleurs['train'])} mẫu | Validation: {len(fleurs['validation'])} mẫu")

## 6. Khởi Tạo Whisper-Large-v3-Turbo (<1.5B) & LoRA

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

MODEL_ID = "openai/whisper-large-v3-turbo"  # 809M tham số

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="vi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Cấu hình LoRA tiết kiệm VRAM cho Colab T4
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 7. Tiền Xử Lý Mel-Spectrogram & Tokenize Nhãn

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    clean_text = prepare_for_wer(batch["sentence"])
    batch["labels"] = processor.tokenizer(clean_text).input_ids
    return batch

print("⚙️ Đang trích xuất đặc trưng âm học (Mel-Spectrogram)... ")
encoded_dataset = fleurs.map(
    prepare_dataset,
    remove_columns=fleurs["train"].column_names,
    num_proc=2
)

## 8. Data Collator & Metric JiWER Đánh Giá WER

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    clean_pred = [prepare_for_wer(s) for s in pred_str]
    clean_label = [prepare_for_wer(s) for s in label_str]
    wer = 100 * wer_metric.compute(predictions=clean_pred, references=clean_label)
    return {"wer": round(wer, 2)}

## 9. Cấu Hình Training Arguments & Bắt Đầu Huấn Luyện 10 Tiếng

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=5000,           # ~8-10 tiếng trên GPU T4
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,       # Giữ lại 3 checkpoint tốt nhất trên Google Drive
    logging_steps=50,
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"]
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("🔥 TIẾN HÀNH HUẤN LUYỆN TỰ ĐỘNG TRONG 10 TIẾNG...")
# Nếu có checkpoint trước đó bị ngắt, có thể truyền resume_from_checkpoint=True
trainer.train(resume_from_checkpoint=False)

## 10. Lưu Trữ Checkpoint & Inference Thử Nghiệm

In [ ]:
best_model_path = f"{CHECKPOINT_DIR}/best_whisper_turbo"
trainer.save_model(best_model_path)
processor.save_pretrained(best_model_path)
print(f"✅ Mô hình đã lưu thành công tại: {best_model_path}")